In [ ]:
pip install --upgrade ultralytics

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Number of CUDA devices: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  Device {i}: {torch.cuda.get_device_name(i)}")
else:
    print("CUDA is not available - check NVIDIA driver")

In [ ]:
from ultralytics import YOLO
import torch

print("YOLO imported successfully!")
print("PyTorch version:", torch.__version__)

# Check and set up GPU
if torch.cuda.is_available():
    torch.cuda.set_device(0)
    torch.cuda.empty_cache()
    print(f" GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = 'cuda'
else:
    print("CUDA not available - falling back to CPU")
    print("   Install PyTorch with CUDA: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
    device = 'cpu'

print(f"Using device: {device}")

In [4]:
import torch
import os

# Clear CUDA cache
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Set CUDA debugging (helps get better error messages)
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Check CUDA is working
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # Test basic CUDA operation
    x = torch.randn(3, 3).cuda()
    print("✓ CUDA working properly")
    

CUDA available: True
CUDA device: NVIDIA GeForce RTX 2060
CUDA memory: 6.44 GB
✓ CUDA working properly


In [5]:
# train_RGB.ipynb
# Cell 1 — imports and sanity check

from ultralytics import YOLO
import torch, os

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version : 2.7.1+cu118
CUDA available  : True
GPU             : NVIDIA GeForce RTX 2060
VRAM            : 6.4 GB


In [6]:
# Cell 2 — verify data.yaml is reachable and class count is correct

import yaml

yaml_path = 'SIMD_RGB/data.yaml'
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

print(f"Train path : {cfg['train']}")
print(f"Val path   : {cfg['val']}")
print(f"Classes    : {cfg['nc']}")
print(f"Names      : {cfg['names']}")

# Count files to confirm
from pathlib import Path
train_count = len(list(Path(cfg['train']).glob('*.jpg')))
val_count   = len(list(Path(cfg['val']).glob('*.jpg')))
print(f"\nTrain images found : {train_count}  (expected: 4000)")
print(f"Val   images found : {val_count}   (expected: 1000)")

Train path : train/images
Val path   : val/images
Classes    : 15
Names      : ['car', 'truck', 'van', 'longvehicle', 'bus', 'airliner', 'propeller', 'trainer', 'chartered', 'fighter', 'other', 'stairtruck', 'pushbacktruck', 'helicopter', 'boat']

Train images found : 0  (expected: 4000)
Val   images found : 0   (expected: 1000)


In [34]:
import os

# Check what's in your SIMD_RGB directory
dataset_path = 'SIMD_RGB'
if os.path.exists(dataset_path):
    print("Contents of SIMD_RGB:")
    for item in os.listdir(dataset_path):
        print(f"  - {item}")
        
    # Check if train/val directories exist
    for folder in ['train', 'val', 'test']:
        folder_path = os.path.join(dataset_path, folder)
        if os.path.exists(folder_path):
            print(f"\n{folder} contents:")
            for subitem in os.listdir(folder_path):
                print(f"  - {subitem}")
else:
    print(f"SIMD_RGB directory not found at {dataset_path}")
    print(f"Current location: {os.getcwd()}")

Contents of SIMD_RGB:
  - data.yaml
  - test
  - train
  - val

train contents:
  - images
  - labels
  - labels.cache

val contents:
  - images
  - labels
  - labels.cache

test contents:
  - images
  - labels
  - labels.cache


In [8]:
# Create the data.yaml file with correct paths
yaml_content = """# SIMD_RGB dataset configuration
path: ./SIMD_RGB  # dataset root directory
train: train/images  # train images (relative to 'path')
val: val/images      # val images (relative to 'path')
test: test/images    # test images (relative to 'path')

# Number of classes - YOU NEED TO UPDATE THIS
nc: 2  # Change this to your actual number of classes

# Class names - YOU NEED TO UPDATE THIS
names: ['class1', 'class2']  # Change these to your actual class names
"""

# Write the file
with open('SIMD_RGB/data.yaml', 'w') as f:
    f.write(yaml_content)
    
print("✓ Created SIMD_RGB/data.yaml")
print("\n⚠️ IMPORTANT: You must update these values in the file:")
print("   - nc: number of classes in your dataset")
print("   - names: your actual class names")

✓ Created SIMD_RGB/data.yaml

⚠️ IMPORTANT: You must update these values in the file:
   - nc: number of classes in your dataset
   - names: your actual class names


In [9]:
import yaml

# Update data.yaml with correct number of classes
with open('SIMD_RGB/data.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Change from 2 to 15 classes
config['nc'] = 15
config['names'] = ['car', 'truck', 'van', 'longvehicle', 'bus', 'airliner', 
                   'propeller', 'trainer', 'chartered', 'fighter', 'other', 
                   'stairtruck', 'pushbacktruck', 'helicopter', 'boat']

# Save the updated config
with open('SIMD_RGB/data.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✓ Updated data.yaml with nc=15 and all class names")
print("\nNow training with all 15 classes!")

✓ Updated data.yaml with nc=15 and all class names

Now training with all 15 classes!


In [12]:
# Run this once before training
import os

cache_files = [
    'SIMD_RGB/train/labels.cache',
    'SIMD_RGB/val/labels.cache',
    'SIMD_RGB/train/labels.cache.npy',
    'SIMD_RGB/val/labels.cache.npy'
]

for cf in cache_files:
    if os.path.exists(cf):
        os.remove(cf)
        print(f"Removed: {cf}")
        
print("Cache cleared. Restart training and warnings should disappear.")

Removed: SIMD_RGB/train/labels.cache
Removed: SIMD_RGB/val/labels.cache
Cache cleared. Restart training and warnings should disappear.


In [13]:
# Cell 3 — train from scratch
# Key argument: pretrained=False  — this is what makes it from-scratch
# Without this, YOLO loads ImageNet-pretrained weights and your
# comparison loses scientific validity.
class_counts = [16241, 2257, 4709, 1352, 1580, 761, 162, 498, 499, 41, 654, 341, 150, 47, 6285]
total = 35577
weights = [total / (15 * c) for c in class_counts]
max_w = max(weights)
weights = [w / max_w for w in weights]

model = YOLO('yolov8n.pt')  # ← This loads pretrained weights
pretrained = False           # ← This tries to override but may not work

results = model.train(
    data       = 'SIMD_RGB/data.yaml',
    epochs     = 100,
    imgsz      = 640,
    batch      = 8,
    lr0        = 0.1,
    optimizer  = 'SGD',
    momentum   = 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    cos_lr     = True,
    augment    = True,
    pretrained = False,
    patience   = 15,
    device     = 0,            # ← GPU (ONE LINE)
    project    = 'runs',
    name       = 'SIMD_RGB',
    exist_ok   = True,
    seed       = 42,
    workers    = 0,
    verbose    = True,
    amp        = False
)
print("\nRGB training complete.")
print(f"Best weights saved to: {results.save_dir}/weights/best.pt")

Ultralytics 8.4.46  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SIMD_RGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=SIMD_RGB, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=15, pers

In [ ]:
# Cell 3 — train from scratch
# Key argument: pretrained=False  — this is what makes it from-scratch
# Without this, YOLO loads ImageNet-pretrained weights and your
# comparison loses scientific validity.
class_counts = [16241, 2257, 4709, 1352, 1580, 761, 162, 498, 499, 41, 654, 341, 150, 47, 6285]
total = 35577
weights = [total / (15 * c) for c in class_counts]
max_w = max(weights)
weights = [w / max_w for w in weights]

model = YOLO('yolov8n.pt')  # ← This loads pretrained weights
pretrained = False           # ← This tries to override but may not work

results = model.train(
    data       = 'SIMD_YGB/data.yaml',
    epochs     = 100,
    imgsz      = 640,
    batch      = 8,
    lr0        = 0.1,
    optimizer  = 'SGD',
    momentum   = 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    cos_lr     = True,
    augment    = True,
    pretrained = False,
    patience   = 15,
    device     = 0,            # ← GPU (ONE LINE)
    project    = 'runs',
    name       = 'SIMD_YGB',
    exist_ok   = True,
    seed       = 42,
    workers    = 0,
    verbose    = True,
    amp        = False
)
print("\nRGB training complete.")
print(f"Best weights saved to: {results.save_dir}/weights/best.pt")

In [30]:
# Cell: Train RGB → YGB → LGB sequentially
from ultralytics import YOLO



# Class weights (same for all datasets)
class_counts = [16241, 2257, 4709, 1352, 1580, 761, 162, 498, 499, 41, 654, 341, 150, 47, 6285]
total = 35577
weights = [total / (15 * c) for c in class_counts]
max_w = max(weights)
weights = [w / max_w for w in weights]

# List of datasets to train
datasets = ['SIMD_LGB']

for dataset in datasets:
    print(f"Training {dataset}...")
    
    # Load model
    model = YOLO('yolov8n.pt')
    
    # Train with your exact parameters
    results = model.train(
        data         = f'{dataset}/data.yaml',
        epochs       = 100,
        imgsz        = 640,
        batch        = 8,
        lr0          = 0.1,
        optimizer    = 'SGD',
        momentum     = 0.937,
        weight_decay = 0.0005,
        warmup_epochs = 3,
        cos_lr       = True,
        augment      = True,
        pretrained   = False,
        patience     = 15,
        device       = 0,
        project      = 'runs',
        name         = dataset,
        exist_ok     = True,
        seed         = 42,
        workers      = 0,
        verbose      = True,
        amp          = False
    )
    
    print(f"\n✅ {dataset} training complete!")
    print(f"Best weights saved to: {results.save_dir}/weights/best.pt")
    print(f"Stopped at epoch: {results.epoch if hasattr(results, 'epoch') else 'N/A'}")

print("ALL THREE TRAININGS COMPLETE!")
print("YGB → LGB all finished successfully!")


Training SIMD_LGB...
Ultralytics 8.4.46  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=SIMD_LGB/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=SIMD_LGB, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=Tr

In [18]:
import os

# Check if directory exists
test_path = r"C:\Users\DELL\Desktop\color\SIMD_RGB\test"
if os.path.exists(test_path):
    print(f"Directory exists: {test_path}")
    
    # List all files in the directory
    files = os.listdir(test_path)
    print(f"Number of files: {len(files)}")
    print(f"First 10 files: {files[:10]}")
    
    # Check for supported image formats
    supported_formats = {'heif', 'bmp', 'jp2', 'jpeg', 'dng', 'tiff', 'tif', 
                         'mpo', 'webp', 'jpg', 'heic', 'png', 'jpeg2000', 'avif'}
    
    image_files = [f for f in files if f.split('.')[-1].lower() in supported_formats]
    print(f"Found {len(image_files)} supported images")
else:
    print(f"Directory does NOT exist: {test_path}")

Directory exists: C:\Users\DELL\Desktop\color\SIMD_RGB\test
Number of files: 3
First 10 files: ['images', 'labels', 'labels.cache']
Found 0 supported images


In [20]:
from ultralytics import YOLO
import yaml
from pathlib import Path
import os

# Define paths
base_path = r"C:\Users\DELL\Desktop\color\SIMD_RGB"
val_path = os.path.join(base_path, "VAL")
test_path = os.path.join(base_path, "test")
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_RGB\weights\best.pt'

# Verify directories exist and count images

print("DATASET VERIFICATION")


for name, path in [("VAL", val_path), ("test", test_path)]:
    images_dir = os.path.join(path, "images")
    if os.path.exists(images_dir):
        image_files = [f for f in os.listdir(images_dir) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'))]
        print(f"{name}: Found {len(image_files)} images in {images_dir}")
        if len(image_files) > 0:
            print(f"  Sample images: {image_files[:3]}")
    else:
        print(f"{name}: Images directory not found at {images_dir}")


print("CREATING YAML CONFIGURATION FILES")


# Common configuration
# Update these based on your actual classes
num_classes = 1  # Change this to your number of classes
class_names = ['object']  # Change this to your actual class names

# Create VAL YAML file
val_yaml = {
    'path': base_path,  # Root directory
    'train': 'train/images',  # Training images (if exists)
    'val': 'VAL/images',      # Validation images (relative to path)
    'test': 'test/images',    # Test images (relative to path)
    'nc': num_classes,         # Number of classes
    'names': class_names,      # Class names
}

# Save VAL YAML
val_yaml_path = 'SIMD_RGB_val.yaml'
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml, f, default_flow_style=False)
print(f"✓ Created {val_yaml_path}")

# Create TEST YAML file
test_yaml = {
    'path': base_path,  # Root directory
    'train': 'train/images',  # Training images (if exists)
    'val': 'VAL/images',      # Validation images (relative to path)
    'test': 'test/images',    # Test images (relative to path)
    'nc': num_classes,         # Number of classes
    'names': class_names,      # Class names
}

# Save TEST YAML
test_yaml_path = 'SIMD_RGB_test.yaml'
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml, f, default_flow_style=False)
print(f"✓ Created {test_yaml_path}")

# Display YAML contents

print("YAML FILE CONTENTS")

print("\n--- SIMD_RGB_val.yaml ---")
with open(val_yaml_path, 'r') as f:
    print(f.read())

print("\n--- SIMD_RGB_test.yaml ---")
with open(test_yaml_path, 'r') as f:
    print(f.read())

# Load the trained model

print("LOADING MODEL")

model = YOLO(model_path)
print(f"✓ Model loaded from {model_path}")

# Run Validation

print("RUNNING VALIDATION ON VAL SET")


try:
    val_metrics = model.val(
        data=val_yaml_path,
        split='val',  # Use validation split
        imgsz=640,
        batch=16,
        device=0,  # Use GPU 0, change to 'cpu' if no GPU
        workers=0,  # Set to 0 to avoid multiprocessing issues
        verbose=True,
        conf=0.25,  # Confidence threshold
        iou=0.45    # IoU threshold for NMS
    )
    
  
    print("RGB VALIDATION RESULTS")

    print(f"   mAP50-95: {val_metrics.box.map:.4f}")
    print(f"   mAP50:     {val_metrics.box.map50:.4f}")
    print(f"   mAP75:     {val_metrics.box.map75:.4f}")
    print(f"   Precision: {val_metrics.box.mp:.4f}")
    print(f"   Recall:    {val_metrics.box.mr:.4f}")
    
except Exception as e:
    print(f"✗ Validation failed: {e}")
    print("\nTroubleshooting tips:")
    print("1. Check if VAL/images directory contains images")
    print("2. Check if VAL/labels directory contains corresponding label files")
    print("3. Ensure label files are in YOLO format (txt files)")
    print("4. Verify that class indices in labels match your YAML configuration")

# Run Testing

print("RUNNING TESTING ON TEST SET")


try:
    test_metrics = model.val(
        data=test_yaml_path,
        split='test',  # Use test split
        imgsz=640,
        batch=16,
        device=0,
        workers=0,
        verbose=True,
        conf=0.25,
        iou=0.45
    )
    

    print("RGB TEST RESULTS")

    print(f"   mAP50-95: {test_metrics.box.map:.4f}")
    print(f"   mAP50:     {test_metrics.box.map50:.4f}")
    print(f"   mAP75:     {test_metrics.box.map75:.4f}")
    print(f"   Precision: {test_metrics.box.mp:.4f}")
    print(f"   Recall:    {test_metrics.box.mr:.4f}")
    
except Exception as e:
    print(f"✗ Testing failed: {e}")
    print("\nTroubleshooting tips:")
    print("1. Check if test/images directory contains images")
    print("2. Check if test/labels directory contains corresponding label files")
    print("3. Ensure label files are in YOLO format (txt files)")

# Optional: Save results to file

print("SAVING RESULTS")


with open('validation_results.txt', 'w') as f:
    f.write("RGB MODEL VALIDATION RESULTS\n")

    f.write(f"mAP50-95: {val_metrics.box.map:.4f}\n")
    f.write(f"mAP50:     {val_metrics.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_metrics.box.map75:.4f}\n")
    f.write(f"Precision: {val_metrics.box.mp:.4f}\n")
    f.write(f"Recall:    {val_metrics.box.mr:.4f}\n\n")
    
    f.write("RGB MODEL TEST RESULTS\n")
    f.write("=" * 30 + "\n")
    f.write(f"mAP50-95: {test_metrics.box.map:.4f}\n")
    f.write(f"mAP50:     {test_metrics.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_metrics.box.map75:.4f}\n")
    f.write(f"Precision: {test_metrics.box.mp:.4f}\n")
    f.write(f"Recall:    {test_metrics.box.mr:.4f}\n")

print("✓ Results saved to 'validation_results.txt'")


print("COMPLETE!")

DATASET VERIFICATION
VAL: Found 500 images in C:\Users\DELL\Desktop\color\SIMD_RGB\VAL\images
  Sample images: ['0011.jpg', '0013.jpg', '0014.jpg']
test: Found 500 images in C:\Users\DELL\Desktop\color\SIMD_RGB\test\images
  Sample images: ['2816.jpg', '2817.jpg', '2818.jpg']
CREATING YAML CONFIGURATION FILES
✓ Created SIMD_RGB_val.yaml
✓ Created SIMD_RGB_test.yaml
YAML FILE CONTENTS

--- SIMD_RGB_val.yaml ---
names:
- object
nc: 1
path: C:\Users\DELL\Desktop\color\SIMD_RGB
test: test/images
train: train/images
val: VAL/images


--- SIMD_RGB_test.yaml ---
names:
- object
nc: 1
path: C:\Users\DELL\Desktop\color\SIMD_RGB
test: test/images
train: train/images
val: VAL/images

LOADING MODEL
✓ Model loaded from C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_RGB\weights\best.pt
RUNNING VALIDATION ON VAL SET
Ultralytics 8.4.46  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast 

In [23]:
from ultralytics import YOLO
import yaml
from pathlib import Path
import os

# Define paths - CHANGE THIS TO YOUR YGB PATH
base_path = r"C:\Users\DELL\Desktop\color\SIMD_YGB"  # ← CHANGED from RGB to YGB
val_path = os.path.join(base_path, "VAL")
test_path = os.path.join(base_path, "test")
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_YGB\weights\best.pt'  # ← CHANGED to YGB

# Verify directories exist and count images

print("DATASET VERIFICATION")


for name, path in [("VAL", val_path), ("test", test_path)]:
    images_dir = os.path.join(path, "images")
    if os.path.exists(images_dir):
        image_files = [f for f in os.listdir(images_dir) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'))]
        print(f"{name}: Found {len(image_files)} images in {images_dir}")
        if len(image_files) > 0:
            print(f"  Sample images: {image_files[:3]}")
    else:
        print(f"{name}: Images directory not found at {images_dir}")


print("CREATING YAML CONFIGURATION FILES")


# Common configuration
# Update these based on your actual classes
num_classes = 1  # Change this to your number of classes
class_names = ['object']  # Change this to your actual class names

# Create VAL YAML file for YGB
val_yaml = {
    'path': base_path,  # Root directory
    'train': 'train/images',  # Training images (if exists)
    'val': 'VAL/images',      # Validation images (relative to path)
    'test': 'test/images',    # Test images (relative to path)
    'nc': num_classes,         # Number of classes
    'names': class_names,      # Class names
}

# Save VAL YAML
val_yaml_path = 'SIMD_YGB_val.yaml'  # ← CHANGED from RGB to YGB
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml, f, default_flow_style=False)
print(f"✓ Created {val_yaml_path}")

# Create TEST YAML file for YGB
test_yaml = {
    'path': base_path,  # Root directory
    'train': 'train/images',  # Training images (if exists)
    'val': 'VAL/images',      # Validation images (relative to path)
    'test': 'test/images',    # Test images (relative to path)
    'nc': num_classes,         # Number of classes
    'names': class_names,      # Class names
}

# Save TEST YAML
test_yaml_path = 'SIMD_YGB_test.yaml'  # ← CHANGED from RGB to YGB
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml, f, default_flow_style=False)
print(f"✓ Created {test_yaml_path}")

# Display YAML contents

print("YAML FILE CONTENTS")

print("\n--- SIMD_YGB_val.yaml ---")  # ← CHANGED
with open(val_yaml_path, 'r') as f:
    print(f.read())

print("\n--- SIMD_YGB_test.yaml ---")  # ← CHANGED
with open(test_yaml_path, 'r') as f:
    print(f.read())

# Load the trained model

print("LOADING YGB MODEL")  # ← CHANGED

model = YOLO(model_path)
print(f"✓ Model loaded from {model_path}")

# Run Validation

print("RUNNING VALIDATION ON YGB VAL SET")  # ← CHANGED


try:
    val_metrics = model.val(
        data=val_yaml_path,
        split='val',  # Use validation split
        imgsz=640,
        batch=16,
        device=0,  # Use GPU 0, change to 'cpu' if no GPU
        workers=0,  # Set to 0 to avoid multiprocessing issues
        verbose=True,
        conf=0.25,  # Confidence threshold
        iou=0.45    # IoU threshold for NMS
    )
    
  
    print("YGB VALIDATION RESULTS")  # ← CHANGED

    print(f"   mAP50-95: {val_metrics.box.map:.4f}")
    print(f"   mAP50:     {val_metrics.box.map50:.4f}")
    print(f"   mAP75:     {val_metrics.box.map75:.4f}")
    print(f"   Precision: {val_metrics.box.mp:.4f}")
    print(f"   Recall:    {val_metrics.box.mr:.4f}")
    
except Exception as e:
    print(f"✗ Validation failed: {e}")
    print("\nTroubleshooting tips:")
    print("1. Check if VAL/images directory contains images")
    print("2. Check if VAL/labels directory contains corresponding label files")
    print("3. Ensure label files are in YOLO format (txt files)")
    print("4. Verify that class indices in labels match your YAML configuration")

# Run Testing

print("RUNNING TESTING ON YGB TEST SET")  # ← CHANGED


try:
    test_metrics = model.val(
        data=test_yaml_path,
        split='test',  # Use test split
        imgsz=640,
        batch=16,
        device=0,
        workers=0,
        verbose=True,
        conf=0.25,
        iou=0.45
    )
    

    print("YGB TEST RESULTS")  # ← CHANGED

    print(f"   mAP50-95: {test_metrics.box.map:.4f}")
    print(f"   mAP50:     {test_metrics.box.map50:.4f}")
    print(f"   mAP75:     {test_metrics.box.map75:.4f}")
    print(f"   Precision: {test_metrics.box.mp:.4f}")
    print(f"   Recall:    {test_metrics.box.mr:.4f}")
    
except Exception as e:
    print(f"✗ Testing failed: {e}")
    print("\nTroubleshooting tips:")
    print("1. Check if test/images directory contains images")
    print("2. Check if test/labels directory contains corresponding label files")
    print("3. Ensure label files are in YOLO format (txt files)")

# Optional: Save results to file

print("SAVING YGB RESULTS")  # ← CHANGED


with open('YGB_validation_results.txt', 'w') as f:  # ← CHANGED filename
    f.write("YGB MODEL VALIDATION RESULTS\n")  # ← CHANGED
    f.write(f"mAP50-95: {val_metrics.box.map:.4f}\n")
    f.write(f"mAP50:     {val_metrics.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_metrics.box.map75:.4f}\n")
    f.write(f"Precision: {val_metrics.box.mp:.4f}\n")
    f.write(f"Recall:    {val_metrics.box.mr:.4f}\n\n")
    
    f.write("YGB MODEL TEST RESULTS\n")  # ← CHANGED
    f.write(f"mAP50-95: {test_metrics.box.map:.4f}\n")
    f.write(f"mAP50:     {test_metrics.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_metrics.box.map75:.4f}\n")
    f.write(f"Precision: {test_metrics.box.mp:.4f}\n")
    f.write(f"Recall:    {test_metrics.box.mr:.4f}\n")

print("✓ YGB results saved to 'YGB_validation_results.txt'")  # ← CHANGED


print("YGB COMPLETE!")  # ← CHANGED

DATASET VERIFICATION
VAL: Found 500 images in C:\Users\DELL\Desktop\color\SIMD_YGB\VAL\images
  Sample images: ['0011.jpg', '0013.jpg', '0014.jpg']
test: Found 500 images in C:\Users\DELL\Desktop\color\SIMD_YGB\test\images
  Sample images: ['2816.jpg', '2817.jpg', '2818.jpg']
CREATING YAML CONFIGURATION FILES
✓ Created SIMD_YGB_val.yaml
✓ Created SIMD_YGB_test.yaml
YAML FILE CONTENTS

--- SIMD_YGB_val.yaml ---
names:
- object
nc: 1
path: C:\Users\DELL\Desktop\color\SIMD_YGB
test: test/images
train: train/images
val: VAL/images


--- SIMD_YGB_test.yaml ---
names:
- object
nc: 1
path: C:\Users\DELL\Desktop\color\SIMD_YGB
test: test/images
train: train/images
val: VAL/images

LOADING YGB MODEL
✓ Model loaded from C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_YGB\weights\best.pt
RUNNING VALIDATION ON YGB VAL SET
Ultralytics 8.4.46  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
va

In [28]:
from ultralytics import YOLO
import yaml
import os
import shutil

# RGB paths
base_path = r"C:\Users\DELL\Desktop\color\SIMD_RGB"
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_RGB\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (15 classes)
num_classes = 15
class_names = ['car', 'truck', 'van', 'longvehicle', 'bus', 
               'airliner', 'propeller', 'trainer', 'chartered', 
               'fighter', 'other', 'stairtruck', 'pushbacktruck', 
               'helicopter', 'boat']

# Create VAL YAML
val_yaml_path = 'SIMD_RGB_val_nc15.yaml'
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML
test_yaml_path = 'SIMD_RGB_test_nc15.yaml'
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)

# Run VALIDATION
print("RUNNING VALIDATION ON RGB")
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("RGB VALIDATION RESULTS (nc=15)")
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# FIXED PER-CLASS RESULTS
print("PER-CLASS RESULTS (mAP50)")
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):  # Fixed: check length instead of using index()
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("RUNNING TESTING ON RGB")
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("RGB TEST RESULTS (nc=15)")
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('RGB_COMPLETE_results_nc15.txt', 'w') as f:
    f.write("RGB MODEL - COMPLETE RESULTS (CORRECT nc=15)\n")
    
    f.write("VALIDATION RESULTS\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")

print("\n✓ Results saved to 'RGB_COMPLETE_results_nc15.txt'")
print("RGB COMPLETE!")

Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_RGB\val\labels.cache
✓ Created: SIMD_RGB_val_nc15.yaml
✓ Created: SIMD_RGB_test_nc15.yaml

RUNNING VALIDATION ON RGB
Ultralytics 8.4.46  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1647.0169.3 MB/s, size: 258.7 KB)
val: Scanning C:\Users\DELL\Desktop\color\SIMD_RGB\val\labels... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 2.5Kit/s 0.2s0.0ss
val: New cache created: C:\Users\DELL\Desktop\color\SIMD_RGB\val\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 3.6it/s 8.9s0.3s
                   all        500       3218      0.703      0.667      0.727      0.579
                   car        230        909      0.639      0.827      0.811      0.604
                 tru

In [29]:
from ultralytics import YOLO
import yaml
import os
import shutil

# YGB paths (CHANGED from RGB)
base_path = r"C:\Users\DELL\Desktop\color\SIMD_YGB"
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_YGB\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (15 classes)
num_classes = 15
class_names = ['car', 'truck', 'van', 'longvehicle', 'bus', 
               'airliner', 'propeller', 'trainer', 'chartered', 
               'fighter', 'other', 'stairtruck', 'pushbacktruck', 
               'helicopter', 'boat']

# Create VAL YAML for YGB
val_yaml_path = 'SIMD_YGB_val_nc15.yaml'  # CHANGED
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML for YGB
test_yaml_path = 'SIMD_YGB_test_nc15.yaml'  # CHANGED
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)
print(f"✓ Model loaded from {model_path}")
print(f"Model classes: {model.names}")
print(f"Number of classes: {len(model.names)}")

# Run VALIDATION
print("RUNNING VALIDATION ON YGB")
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("YGB VALIDATION RESULTS (nc=15)")
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# PER-CLASS RESULTS
print("PER-CLASS RESULTS (mAP50)")
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("RUNNING TESTING ON YGB")
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("YGB TEST RESULTS (nc=15)")
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('YGB_COMPLETE_results_nc15.txt', 'w') as f:  # CHANGED filename
    f.write("YGB MODEL - COMPLETE RESULTS (CORRECT nc=15)\n")  # CHANGED
    
    f.write("VALIDATION RESULTS\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")

print("\n✓ Results saved to 'YGB_COMPLETE_results_nc15.txt'")
print("YGB COMPLETE!")


Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_YGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_YGB\test\labels.cache
✓ Created: SIMD_YGB_val_nc15.yaml
✓ Created: SIMD_YGB_test_nc15.yaml
✓ Model loaded from C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_YGB\weights\best.pt
Model classes: {0: 'car', 1: 'truck', 2: 'van', 3: 'longvehicle', 4: 'bus', 5: 'airliner', 6: 'propeller', 7: 'trainer', 8: 'chartered', 9: 'fighter', 10: 'other', 11: 'stairtruck', 12: 'pushbacktruck', 13: 'helicopter', 14: 'boat'}
Number of classes: 15

RUNNING VALIDATION ON YGB
Ultralytics 8.4.46  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1144.8183.7 MB/s, size: 218.4 KB)
val: Scanning C:\Users\DELL\Desktop\color\SIMD_YGB\val\labels... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 2.4Kit/s 0.2s0.

In [33]:
from ultralytics import YOLO
import yaml
import os
import shutil

# LGB paths (CHANGED from YGB to LGB)
base_path = r"C:\Users\DELL\Desktop\color\SIMD_LGB"
model_path = r'C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_LGB\weights\best.pt'

# DELETE OLD CACHE FILES
print("Cleaning old cache files...")
cache_files = [
    os.path.join(base_path, "val", "labels.cache"),
    os.path.join(base_path, "test", "labels.cache"),
]
for cache_file in cache_files:
    if os.path.exists(cache_file):
        os.remove(cache_file)
        print(f"✓ Deleted: {cache_file}")

# CORRECT class configuration (15 classes)
num_classes = 15
class_names = ['car', 'truck', 'van', 'longvehicle', 'bus', 
               'airliner', 'propeller', 'trainer', 'chartered', 
               'fighter', 'other', 'stairtruck', 'pushbacktruck', 
               'helicopter', 'boat']

# Create VAL YAML for LGB
val_yaml_path = 'SIMD_LGB_val_nc15.yaml'  # CHANGED to LGB
val_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(val_yaml_path, 'w') as f:
    yaml.dump(val_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {val_yaml_path}")

# Create TEST YAML for LGB
test_yaml_path = 'SIMD_LGB_test_nc15.yaml'  # CHANGED to LGB
test_yaml_content = {
    'path': base_path,
    'train': 'train/images',
    'val': 'VAL/images',
    'test': 'test/images',
    'nc': num_classes,
    'names': class_names,
}
with open(test_yaml_path, 'w') as f:
    yaml.dump(test_yaml_content, f, default_flow_style=False)
print(f"✓ Created: {test_yaml_path}")

# Load model
model = YOLO(model_path)
print(f"✓ Model loaded from {model_path}")
print(f"Model classes: {model.names}")
print(f"Number of classes: {len(model.names)}")

# Run VALIDATION
print("RUNNING VALIDATION ON LGB")
val_results = model.val(
    data=val_yaml_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("LGB VALIDATION RESULTS (nc=15)")
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP75:     {val_results.box.map75:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

# PER-CLASS RESULTS
print("PER-CLASS RESULTS (mAP50)")
for i, name in enumerate(class_names):
    if i < len(val_results.box.ap50):
        print(f"{name:20} mAP50: {val_results.box.ap50[i]:.4f}")

# Run TESTING
print("RUNNING TESTING ON LGB")
test_results = model.val(
    data=test_yaml_path,
    split='test',
    imgsz=640,
    batch=16,
    device=0,
    workers=0,
    verbose=True
)

print("LGB TEST RESULTS (nc=15)")
print(f"mAP50-95: {test_results.box.map:.4f}")
print(f"mAP50:     {test_results.box.map50:.4f}")
print(f"mAP75:     {test_results.box.map75:.4f}")
print(f"Precision: {test_results.box.mp:.4f}")
print(f"Recall:    {test_results.box.mr:.4f}")

# Save results
with open('LGB_COMPLETE_results_nc15.txt', 'w') as f:  # CHANGED filename to LGB
    f.write("LGB MODEL - COMPLETE RESULTS (CORRECT nc=15)\n")  # CHANGED to LGB
    
    f.write("VALIDATION RESULTS\n")
    f.write(f"mAP50-95: {val_results.box.map:.4f}\n")
    f.write(f"mAP50:     {val_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {val_results.box.map75:.4f}\n")
    f.write(f"Precision: {val_results.box.mp:.4f}\n")
    f.write(f"Recall:    {val_results.box.mr:.4f}\n\n")
    
    f.write("TEST RESULTS\n")
    f.write(f"mAP50-95: {test_results.box.map:.4f}\n")
    f.write(f"mAP50:     {test_results.box.map50:.4f}\n")
    f.write(f"mAP75:     {test_results.box.map75:.4f}\n")
    f.write(f"Precision: {test_results.box.mp:.4f}\n")
    f.write(f"Recall:    {test_results.box.mr:.4f}\n")

print("\n✓ Results saved to 'LGB_COMPLETE_results_nc15.txt'")
print("LGB COMPLETE!")

Cleaning old cache files...
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_LGB\val\labels.cache
✓ Deleted: C:\Users\DELL\Desktop\color\SIMD_LGB\test\labels.cache
✓ Created: SIMD_LGB_val_nc15.yaml
✓ Created: SIMD_LGB_test_nc15.yaml
✓ Model loaded from C:\Users\DELL\Desktop\color\runs\detect\runs\SIMD_LGB\weights\best.pt
Model classes: {0: 'car', 1: 'truck', 2: 'van', 3: 'longvehicle', 4: 'bus', 5: 'airliner', 6: 'propeller', 7: 'trainer', 8: 'chartered', 9: 'fighter', 10: 'other', 11: 'stairtruck', 12: 'pushbacktruck', 13: 'helicopter', 14: 'boat'}
Number of classes: 15

RUNNING VALIDATION ON LGB
Ultralytics 8.4.46  Python-3.13.9 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
Model summary (fused): 73 layers, 3,008,573 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 692.2160.7 MB/s, size: 252.4 KB)
val: Scanning C:\Users\DELL\Desktop\color\SIMD_LGB\val\labels... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 2.5Kit/s 0.2s<0.